# Task 3: Symmetric vs. Asymmetric INT8 Quantization

## Objective

Implement and compare **symmetric** and **asymmetric** INT8 quantization. Understand why **weights** (centered around zero) and **activations** (often non-negative) benefit from different quantization methods.

---

## Input

```python
weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)
```

---

# Part A: Symmetric INT8 Quantization

### Quantized Range

- **INT8 Range:** `[-127, 127]`
- **Zero Point:** `0`
- **Scale Formula:**

```text
scale = max(|x_min|, |x_max|) / 127
```

### Quantization Formula

```text
q = round(x / scale)
```

Clip the result to `[-127, 127]` and return as `np.int8`.

```python
def symmetric_quantize(tensor):
    """
    Returns:
        quantized_tensor (np.int8)
        scale (float)
        zero_point = 0
    """
    pass
```

---

# Part B: Asymmetric INT8 Quantization

### Quantized Range

- **INT8 Range:** `[-128, 127]`

### Scale

```text
scale = (x_max - x_min) / 255
```

### Zero Point

```text
zero_point = round(-128 - x_min / scale)
```

### Quantization

```text
q = round(x / scale) + zero_point
```

Clip both **zero point** and **q** to `[-128, 127]`, then return as `np.int8`.

```python
def asymmetric_quantize(tensor):
    """
    Returns:
        quantized_tensor (np.int8)
        scale (float)
        zero_point (int)
    """
    pass
```

---

# Part C: Dequantization

### Formula

```text
x_dequantized = (q - zero_point) × scale
```

```python
def dequantize(quantized_tensor, scale, zero_point):
    """
    Returns:
        Dequantized float tensor
    """
    pass
```

---

# Part D: Apply Both Methods and Compare

Apply both quantization methods separately to:

- Weights
- Activations

Fill in the following comparison table.

| Tensor | Method | Scale | Zero Pt | MAE | MSE | Max Err | Sat(min) | Sat(max) | Sat(total) |
|---------|--------|------:|---------:|----:|----:|--------:|----------:|----------:|------------:|
| Weights | Symmetric | | | | | | | | |
| Weights | Asymmetric | | | | | | | | |
| Activations | Symmetric | | | | | | | | |
| Activations | Asymmetric | | | | | | | | |

---

# Part E: Compare Reconstructed Values

For both tensors, display side by side:

- Original tensor
- Symmetrically quantized (INT8)
- Symmetrically dequantized (float)
- Asymmetrically quantized (INT8)
- Asymmetrically dequantized (float)

---

# Part F: Outlier Experiment

```python
outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)
```

## 1. Apply Both Quantization Methods

Report:

- Scale
- Zero Point
- MAE
- MSE
- Max Error
- Per-element Error

---

## 2. Remove the Outlier

Repeat the experiment after removing **12.0**.

```python
[-0.5, -0.2, 0.0, 0.3, 0.7]
```

Again report:

- Scale
- Zero Point
- MAE
- MSE
- Max Error
- Per-element Error

---

## 3. Compare the Results

Fill in the following table.

| Version | Method | Scale | Zero Pt | MAE | MSE | Max Error |
|----------|--------|------:|---------:|----:|----:|----------:|
| With outlier | Symmetric | | | | | |
| With outlier | Asymmetric | | | | | |
| Without outlier | Symmetric | | | | | |
| Without outlier | Asymmetric | | | | | |

---

# Deliverable

Your notebook should include:

- ✅ Implementation of symmetric quantization
- ✅ Implementation of asymmetric quantization
- ✅ Dequantization function
- ✅ Comparison table for weights and activations
- ✅ Side-by-side reconstructed tensors
- ✅ Outlier experiment
- ✅ Comparison table for the outlier experiment
- ✅ Written observations discussing:
  - When symmetric quantization performs better
  - When asymmetric quantization performs better
  - Effect of outliers on quantization quality
  - Why weights typically use symmetric quantization
  - Why activations often use asymmetric quantization



In [1]:
import torch
import pandas as pd

In [2]:
weights = torch.tensor([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=torch.float32)

activations = torch.tensor([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=torch.float32)

outlier_tensor = torch.tensor(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=torch.float32
)

In [3]:
def symmetric_quantize(tensor):

    qmin = -127
    qmax = 127

    max_abs = torch.max(torch.abs(tensor)).item()

    if max_abs == 0:
        scale = 1.0
    else:
        scale = max_abs / qmax

    zero_point = 0

    q = torch.round(tensor / scale)
    q = torch.clamp(q, qmin, qmax)

    return q.to(torch.int8), scale, zero_point

In [4]:
def asymmetric_quantize(tensor):

    qmin = -128
    qmax = 127

    xmin = tensor.min().item()
    xmax = tensor.max().item()

    if xmax == xmin:
        scale = 1.0
        zero_point = 0
    else:
        scale = (xmax - xmin) / (qmax - qmin)

        zero_point = round(qmin - xmin / scale)
        zero_point = max(qmin, min(qmax, zero_point))

    q = torch.round(tensor / scale) + zero_point
    q = torch.clamp(q, qmin, qmax)

    return q.to(torch.int8), scale, zero_point

In [5]:
def dequantize(q_tensor, scale, zero_point):

    return (q_tensor.float() - zero_point) * scale

In [6]:
def calculate_metrics(original, quantized, dequantized):

    error = original - dequantized

    abs_error = torch.abs(error)

    mae = abs_error.mean().item()

    mse = (error ** 2).mean().item()

    max_error = abs_error.max().item()

    sat_min = (quantized == -128).sum().item() + (quantized == -127).sum().item()

    sat_max = (quantized == 127).sum().item()

    sat_total = sat_min + sat_max

    return {
        "MAE": mae,
        "MSE": mse,
        "Max Error": max_error,
        "Sat(min)": sat_min,
        "Sat(max)": sat_max,
        "Sat(total)": sat_total
    }

In [7]:
def compare_methods(name, tensor):

    print("=" * 70)
    print(name)
    print("=" * 70)

    q_sym, scale_sym, zp_sym = symmetric_quantize(tensor)
    dq_sym = dequantize(q_sym, scale_sym, zp_sym)

    q_asym, scale_asym, zp_asym = asymmetric_quantize(tensor)
    dq_asym = dequantize(q_asym, scale_asym, zp_asym)

    print("\nOriginal")
    print(tensor)

    print("\nSymmetric Quantized")
    print(q_sym)

    print("\nSymmetric Dequantized")
    print(dq_sym)

    print("\nAsymmetric Quantized")
    print(q_asym)

    print("\nAsymmetric Dequantized")
    print(dq_asym)

    sym_metrics = calculate_metrics(tensor, q_sym, dq_sym)
    asym_metrics = calculate_metrics(tensor, q_asym, dq_asym)

    return [
        {
            "Tensor": name,
            "Method": "Symmetric",
            "Scale": scale_sym,
            "Zero Pt": zp_sym,
            **sym_metrics
        },
        {
            "Tensor": name,
            "Method": "Asymmetric",
            "Scale": scale_asym,
            "Zero Pt": zp_asym,
            **asym_metrics
        }
    ]

In [8]:
results = []

results.extend(compare_methods("Weights", weights))
results.extend(compare_methods("Activations", activations))

df = pd.DataFrame(results)

print("\nComparison Table\n")
print(df)

Weights

Original
tensor([[-1.8000, -0.9000,  0.0000,  0.7000,  1.5000],
        [-2.4000, -0.3000,  0.2000,  1.1000,  2.0000]])

Symmetric Quantized
tensor([[ -95,  -48,    0,   37,   79],
        [-127,  -16,   11,   58,  106]], dtype=torch.int8)

Symmetric Dequantized
tensor([[-1.7953, -0.9071,  0.0000,  0.6992,  1.4929],
        [-2.4000, -0.3024,  0.2079,  1.0961,  2.0031]])

Asymmetric Quantized
tensor([[ -93,  -41,   11,   52,   98],
        [-128,   -6,   23,   75,  127]], dtype=torch.int8)

Asymmetric Dequantized
tensor([[-1.7945, -0.8973,  0.0000,  0.7075,  1.5012],
        [-2.3984, -0.2933,  0.2071,  1.1043,  2.0016]])
Activations

Original
tensor([[0.0000, 0.3000, 0.8000, 1.4000, 2.1000],
        [0.1000, 0.6000, 1.0000, 1.8000, 3.2000]])

Symmetric Quantized
tensor([[  0,  12,  32,  56,  83],
        [  4,  24,  40,  71, 127]], dtype=torch.int8)

Symmetric Dequantized
tensor([[0.0000, 0.3024, 0.8063, 1.4110, 2.0913],
        [0.1008, 0.6047, 1.0079, 1.7890, 3.2000]])

Asy

In [9]:
def run_outlier_case(name, tensor):

    rows = []

    print("\n", "="*60)
    print(name)
    print("="*60)

    q_sym, s_sym, zp_sym = symmetric_quantize(tensor)
    dq_sym = dequantize(q_sym, s_sym, zp_sym)

    error_sym = torch.abs(tensor-dq_sym)

    print("\nSymmetric")
    print("Scale:", s_sym)
    print("Zero Point:", zp_sym)
    print("Per-element Error:", error_sym)

    rows.append({
        "Version": name,
        "Method": "Symmetric",
        "Scale": s_sym,
        "Zero Pt": zp_sym,
        **calculate_metrics(tensor,q_sym,dq_sym)
    })

    q_asym,s_asym,zp_asym = asymmetric_quantize(tensor)
    dq_asym = dequantize(q_asym,s_asym,zp_asym)

    error_asym = torch.abs(tensor-dq_asym)

    print("\nAsymmetric")
    print("Scale:", s_asym)
    print("Zero Point:", zp_asym)
    print("Per-element Error:", error_asym)

    rows.append({
        "Version": name,
        "Method": "Asymmetric",
        "Scale": s_asym,
        "Zero Pt": zp_asym,
        **calculate_metrics(tensor,q_asym,dq_asym)
    })

    return rows

In [10]:
outlier_results = []

outlier_results.extend(
    run_outlier_case(
        "With Outlier",
        outlier_tensor
    )
)

outlier_results.extend(
    run_outlier_case(
        "Without Outlier",
        outlier_tensor[:-1]
    )
)

outlier_df = pd.DataFrame(outlier_results)

print("\nOutlier Comparison\n")
print(outlier_df)


With Outlier

Symmetric
Scale: 0.09448818897637795
Zero Point: 0
Per-element Error: tensor([0.0276, 0.0110, 0.0000, 0.0165, 0.0386, 0.0000])

Asymmetric
Scale: 0.049019607843137254
Zero Point: -118
Per-element Error: tensor([0.0098, 0.0039, 0.0000, 0.0059, 0.0137, 0.0098])

Without Outlier

Symmetric
Scale: 0.005511810929756465
Zero Point: 0
Per-element Error: tensor([0.0016, 0.0016, 0.0000, 0.0024, 0.0000])

Asymmetric
Scale: 0.004705882306192436
Zero Point: -22
Per-element Error: tensor([0.0012, 0.0024, 0.0000, 0.0012, 0.0012])

Outlier Comparison

           Version      Method     Scale  Zero Pt       MAE       MSE  \
0     With Outlier   Symmetric  0.094488        0  0.015617  0.000441   
1     With Outlier  Asymmetric  0.049020     -118  0.007190  0.000072   
2  Without Outlier   Symmetric  0.005512        0  0.001102  0.000002   
3  Without Outlier  Asymmetric  0.004706      -22  0.001176  0.000002   

   Max Error  Sat(min)  Sat(max)  Sat(total)  
0   0.038583         0       